# Phase 1 implementation

Canonical transformation and explicit optional raw duplicate audits. Run all cells in a fresh kernel. General transformation does not read or change monitoring configuration. Phase 2/3 prepare their own cohort threshold through the shared engine.


In [ ]:
#!/usr/bin/env python3
"""
Phase 1 transformation runner for the Distributed Monitoring POC.

What this script does:
1. Reads raw CSV telemetry files from Data/files_csv.
2. Applies a transparent Phase 1 transformation (Temperature only).
3. Produces reviewable artifacts with explicit audit/QC information.
4. Enforces configurable quality gates from config/transformation_defaults.yaml.

Why this script exists:
- The project requires data preparation to be traceable and reviewable.
- Every major transformation step records counts and decisions.
- Output artifacts are deterministic for reproducibility.
"""

from __future__ import annotations

import argparse
import csv
import datetime as dt
import hashlib
import json
import math
import re
import statistics
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# -----------------------------------------------------------------------------
# Utility helpers
# -----------------------------------------------------------------------------


def _now_utc_iso() -> str:
    """Return current UTC time in ISO format for logs and decisions metadata."""
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def _sha256_file(path: Path) -> str:
    """Hash a file for reproducibility checks."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _parse_filename_timestamp(name: str) -> Optional[dt.datetime]:
    """
    Parse timestamps embedded in filenames like:
      2022_02_11_13-45-52_data.csv
      2022_06_13_00-00-19.csv

    Returns timezone-aware UTC datetime when parseable, else None.
    """
    m = re.search(r"(\d{4}_\d{2}_\d{2}_\d{2}-\d{2}-\d{2})", name)
    if not m:
        return None
    try:
        ts = dt.datetime.strptime(m.group(1), "%Y_%m_%d_%H-%M-%S")
        return ts.replace(tzinfo=dt.timezone.utc)
    except ValueError:
        return None


def _bucket_epoch(ts_epoch: int, bucket_size_seconds: int) -> int:
    """Map a unix-epoch second to a fixed bucket boundary."""
    return ts_epoch - (ts_epoch % bucket_size_seconds)


def _safe_float(v: str) -> Optional[float]:
    """Best-effort conversion to float. Returns None on invalid values."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _safe_int_epoch(v: str) -> Optional[int]:
    """
    Parse epoch time that may be integer-like or float-like text.
    Example: "1644587153" or "1644587153.0".
    """
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None


def _safe_numeric(v: Any) -> Optional[float]:
    """Best-effort numeric conversion for mixed row values."""
    if v is None or v == "":
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _percentile_linear(sorted_values: List[float], p: int) -> Optional[float]:
    """
    Compute percentile using linear interpolation between nearest ranks.
    p is expected in [0, 100].
    """
    if not sorted_values:
        return None
    if p <= 0:
        return sorted_values[0]
    if p >= 100:
        return sorted_values[-1]

    n = len(sorted_values)
    pos = (p / 100.0) * (n - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))

    if lo == hi:
        return sorted_values[lo]

    weight = pos - lo
    return sorted_values[lo] * (1.0 - weight) + sorted_values[hi] * weight


# -----------------------------------------------------------------------------
# Config loading
# -----------------------------------------------------------------------------


def load_yaml_config(path: Path) -> Dict[str, Any]:
    """
    Load the transformation config.

    PyYAML is used when available, but the script also includes a small
    standard-library fallback so it can run on a plain Windows Python install
    without any extra packages.
    """
    try:
        import yaml  # type: ignore
    except ImportError:
        yaml = None

    if yaml is not None:
        with path.open("r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
    else:
        cfg = _load_yaml_subset(path)

    if not isinstance(cfg, dict):
        raise ValueError(f"Config at {path} is not a mapping/object.")
    return cfg


def _load_yaml_subset(path: Path) -> Dict[str, Any]:
    """
    Parse the small YAML subset used by the project config.

    Supported forms:
    - top-level and nested mappings
    - list items introduced by '-'
    - strings, integers, floats, booleans, and quoted strings

    This keeps the script dependency-free while still reading the existing
    config file format.
    """

    def parse_scalar(text: str) -> Any:
        value = text.strip()
        if value == "":
            return ""
        if value.startswith('"') and value.endswith('"') and len(value) >= 2:
            return value[1:-1]
        if value.startswith("'") and value.endswith("'") and len(value) >= 2:
            return value[1:-1]
        lowered = value.lower()
        if lowered == "true":
            return True
        if lowered == "false":
            return False
        if lowered in {"null", "none", "~"}:
            return None
        if re.fullmatch(r"[-+]?\d+", value):
            try:
                return int(value)
            except ValueError:
                pass
        if re.fullmatch(r"[-+]?\d*\.\d+", value):
            try:
                return float(value)
            except ValueError:
                pass
        return value

    def leading_spaces(line: str) -> int:
        return len(line) - len(line.lstrip(" "))

    lines: List[Tuple[int, str]] = []
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        stripped = raw_line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        if "#" in raw_line:
            hash_index = raw_line.find("#")
            if hash_index > 0 and raw_line[:hash_index].rstrip().endswith(("'", '"')):
                pass
            else:
                raw_line = raw_line[:hash_index]
        lines.append((leading_spaces(raw_line), raw_line.rstrip()))

    if not lines:
        return {}

    def parse_block(index: int, indent: int) -> Tuple[Any, int]:
        result: Any = None
        while index < len(lines):
            current_indent, raw_line = lines[index]
            stripped = raw_line.strip()
            if current_indent < indent:
                break
            if stripped.startswith("-") and current_indent == indent:
                if result is None:
                    result = []
                elif not isinstance(result, list):
                    raise ValueError(f"Mixed mapping/list structure near: {raw_line}")
                item_text = stripped[1:].strip()
                index += 1
                if item_text == "":
                    if index < len(lines) and lines[index][0] > current_indent:
                        item_value, index = parse_block(index, lines[index][0])
                    else:
                        item_value = {}
                else:
                    item_value = parse_scalar(item_text)
                    if index < len(lines) and lines[index][0] > current_indent:
                        nested_value, index = parse_block(index, lines[index][0])
                        if isinstance(item_value, dict):
                            if isinstance(nested_value, dict):
                                item_value.update(nested_value)
                            else:
                                raise ValueError(
                                    f"List item mapping expected near: {raw_line}"
                                )
                        else:
                            if isinstance(nested_value, dict):
                                item_value = {"value": item_value, **nested_value}
                            else:
                                item_value = nested_value
                result.append(item_value)
                continue

            if ":" not in stripped:
                raise ValueError(f"Unsupported config line: {raw_line}")

            if result is None:
                result = {}
            elif not isinstance(result, dict):
                raise ValueError(f"Mixed mapping/list structure near: {raw_line}")

            key, value_text = stripped.split(":", 1)
            key = key.strip()
            value_text = value_text.strip()
            index += 1

            if value_text == "":
                if index < len(lines) and lines[index][0] > current_indent:
                    nested_value, index = parse_block(index, lines[index][0])
                    result[key] = nested_value
                else:
                    result[key] = {}
            else:
                result[key] = parse_scalar(value_text)
                if index < len(lines) and lines[index][0] > current_indent:
                    nested_value, index = parse_block(index, lines[index][0])
                    if isinstance(result[key], dict):
                        if isinstance(nested_value, dict):
                            result[key].update(nested_value)
                        else:
                            raise ValueError(f"Mapping expected near: {raw_line}")
                    else:
                        result[key] = nested_value

        if result is None:
            result = {}
        return result, index

    parsed, final_index = parse_block(0, lines[0][0])
    if final_index != len(lines):
        raise ValueError(f"Could not fully parse config file {path}")
    if not isinstance(parsed, dict):
        raise ValueError(f"Config at {path} is not a mapping/object.")
    return parsed


# -----------------------------------------------------------------------------
# Core transformation
# -----------------------------------------------------------------------------


NON_SOURCE_CSV_FILENAMES = frozenset({"temperature_raw.csv"})


def discover_csv_files(data_dir: Path, limit_files: Optional[int]) -> List[Path]:
    """
    Discover CSV files under data directory and sort by timestamp-in-filename
    when present, fallback to lexical name order. Exclude known aggregate CSVs
    before applying the optional source-file limit.
    """
    files = [
        p
        for p in data_dir.glob("*.csv")
        if p.is_file()
        and p.name.lower().endswith(".csv")
        and p.name.lower() not in NON_SOURCE_CSV_FILENAMES
    ]

    files.sort(
        key=lambda p: (
            (
                _parse_filename_timestamp(p.name)
                or dt.datetime.min.replace(tzinfo=dt.timezone.utc)
            ),
            p.name,
        )
    )

    if limit_files is not None:
        files = files[:limit_files]
    return files


def build_phase1_temperature(
    files: List[Path],
    bucket_size_seconds: int,
    stale_threshold_seconds: int,
    min_devices_per_snapshot: int,
    drop_snapshot_if_stale_over_threshold: bool,
    completeness_weight: float,
    freshness_weight: float,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:
    """
    Execute full Phase 1 transformation and return:
    1) raw_index_rows: per-file coverage and parsing audit
    2) aligned_rows: final aligned snapshot rows for output CSV
    3) qc: summarized quality metrics for gates and review

    Design notes:
    - We keep full trace counters for auditability.
    - Deduplication is performed for temperature stream keys used in Phase 1.
    - Alignment is observed-only: missing device values remain blank and act as a
      coverage penalty instead of being filled or inferred.
    """

    raw_index_rows: List[Dict[str, Any]] = []
    counters = {
        "files_processed": 0,
        "rows_total": 0,
        "rows_schema_valid": 0,
        "rows_schema_invalid": 0,
        "rows_temperature": 0,
        "rows_temperature_valid": 0,
        "rows_temperature_invalid": 0,
        "rows_temperature_duplicate": 0,
    }

    values_by_bucket_device: Dict[Tuple[int, str], List[float]] = defaultdict(list)
    seen_temp_keys: set[Tuple[int, str, str]] = set()
    device_ids_seen: set[str] = set()

    for file_path in files:
        counters["files_processed"] += 1

        row_count = 0
        schema_valid = 0
        schema_invalid = 0
        ts_min: Optional[int] = None
        ts_max: Optional[int] = None

        with file_path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)

            expected = {"Time", "DeviceId", "Sensor", "Value"}
            fieldnames = set(reader.fieldnames or [])
            schema_ok = expected.issubset(fieldnames)

            if not schema_ok:
                for _ in reader:
                    row_count += 1
                    schema_invalid += 1
                    counters["rows_total"] += 1
                    counters["rows_schema_invalid"] += 1

                raw_index_rows.append(
                    {
                        "file_name": file_path.name,
                        "file_path": str(file_path),
                        "file_timestamp_utc": (
                            _parse_filename_timestamp(file_path.name) or ""
                        ),
                        "row_count": row_count,
                        "schema_valid_rows": 0,
                        "schema_invalid_rows": schema_invalid,
                        "min_time_epoch": "",
                        "max_time_epoch": "",
                        "status": "invalid_schema",
                    }
                )
                continue

            for row in reader:
                row_count += 1
                counters["rows_total"] += 1

                t = row.get("Time", "")
                d = row.get("DeviceId", "")
                s = row.get("Sensor", "")
                v = row.get("Value", "")

                ts_epoch = _safe_int_epoch(t)
                val = _safe_float(v)

                if ts_epoch is None or not d or not s or val is None:
                    schema_invalid += 1
                    counters["rows_schema_invalid"] += 1
                    continue

                schema_valid += 1
                counters["rows_schema_valid"] += 1

                ts_min = ts_epoch if ts_min is None else min(ts_min, ts_epoch)
                ts_max = ts_epoch if ts_max is None else max(ts_max, ts_epoch)

                if s != "Temperature":
                    continue

                counters["rows_temperature"] += 1

                temp_key = (ts_epoch, d, s)
                if temp_key in seen_temp_keys:
                    counters["rows_temperature_duplicate"] += 1
                    continue
                seen_temp_keys.add(temp_key)

                counters["rows_temperature_valid"] += 1
                device_ids_seen.add(d)

                bucket = _bucket_epoch(ts_epoch, bucket_size_seconds)
                values_by_bucket_device[(bucket, d)].append(val)

        raw_index_rows.append(
            {
                "file_name": file_path.name,
                "file_path": str(file_path),
                "file_timestamp_utc": (_parse_filename_timestamp(file_path.name) or ""),
                "row_count": row_count,
                "schema_valid_rows": schema_valid,
                "schema_invalid_rows": schema_invalid,
                "min_time_epoch": "" if ts_min is None else ts_min,
                "max_time_epoch": "" if ts_max is None else ts_max,
                "status": "ok",
            }
        )

    agg_value_by_bucket_device: Dict[Tuple[int, str], float] = {}
    for key, vals in values_by_bucket_device.items():
        agg_value_by_bucket_device[key] = float(statistics.median(vals))

    buckets_sorted = sorted({b for (b, _) in agg_value_by_bucket_device.keys()})
    devices_sorted = sorted(device_ids_seen)

    aligned_rows: List[Dict[str, Any]] = []
    total_cell_count = 0
    stale_snapshot_count = 0

    for bucket in buckets_sorted:
        row: Dict[str, Any] = {
            "bucket_epoch": bucket,
            "bucket_time_utc": dt.datetime.fromtimestamp(
                bucket, tz=dt.timezone.utc
            ).isoformat(),
        }

        available_devices = 0
        max_age_seconds = 0
        avg_candidates_all_devices: List[float] = []

        for device in devices_sorted:
            total_cell_count += 1
            direct_key = (bucket, device)
            value: Optional[float] = None

            if direct_key in agg_value_by_bucket_device:
                value = agg_value_by_bucket_device[direct_key]

            row[f"{device}_temperature"] = "" if value is None else round(value, 6)

            if value is not None:
                avg_candidates_all_devices.append(value)
                available_devices += 1

        total_fields = len(devices_sorted) if devices_sorted else 1
        completeness_component = (
            (available_devices / total_fields) if total_fields else 0.0
        )
        freshness_component = 1.0 - min(
            1.0, max_age_seconds / max(1, stale_threshold_seconds)
        )
        confidence = (
            completeness_weight * completeness_component
            + freshness_weight * freshness_component
        )

        row["available_devices"] = available_devices
        row["max_age_seconds"] = max_age_seconds
        row["snapshot_confidence"] = round(confidence, 6)
        row["meets_min_device_coverage"] = int(
            available_devices >= min_devices_per_snapshot
        )
        row["average_temperature_all_devices"] = (
            ""
            if not avg_candidates_all_devices
            else round(
                sum(avg_candidates_all_devices) / len(avg_candidates_all_devices), 6
            )
        )
        row["average_temperature_all_devices_count"] = len(avg_candidates_all_devices)

        if available_devices < min_devices_per_snapshot:
            stale_snapshot_count += 1

        aligned_rows.append(row)

    schema_validity_percent = (
        100.0 * counters["rows_schema_valid"] / max(1, counters["rows_total"])
    )
    duplicate_key_rate_percent = (
        100.0
        * counters["rows_temperature_duplicate"]
        / max(1, counters["rows_temperature"])
    )
    stale_snapshot_rate_percent = (
        100.0 * stale_snapshot_count / max(1, len(aligned_rows))
    )
    coverage_ok_count = sum(
        1 for r in aligned_rows if r["meets_min_device_coverage"] == 1
    )
    device_coverage_percent = 100.0 * coverage_ok_count / max(1, len(aligned_rows))

    qc = {
        **counters,
        "devices_seen_count": len(devices_sorted),
        "snapshot_count": len(aligned_rows),
        "schema_validity_percent": round(schema_validity_percent, 6),
        "duplicate_key_rate_percent": round(duplicate_key_rate_percent, 6),
        "stale_snapshot_rate_percent": round(stale_snapshot_rate_percent, 6),
        "device_coverage_percent": round(device_coverage_percent, 6),
    }

    return raw_index_rows, aligned_rows, qc


# -----------------------------------------------------------------------------
# Artifact writers and quality gate checks
# -----------------------------------------------------------------------------


def write_csv_rows(path: Path, rows: List[Dict[str, Any]]) -> None:
    """Write list-of-dict rows to CSV with deterministic field ordering."""
    path.parent.mkdir(parents=True, exist_ok=True)

    if not rows:
        path.write_text("", encoding="utf-8")
        return

    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_qc_report(path: Path, qc: Dict[str, Any]) -> None:
    """Write single-row QC report as CSV for easy downstream review."""
    csv_qc = dict(qc)
    if "excluded_non_source_csv_filenames" in csv_qc:
        csv_qc["excluded_non_source_csv_filenames"] = json.dumps(
            csv_qc["excluded_non_source_csv_filenames"]
        )
    write_csv_rows(path, [csv_qc])


def build_phase1_average_statistics_rows(
    aligned_rows: List[Dict[str, Any]],
    confidence_threshold: float,
    percentiles: List[int],
) -> List[Dict[str, Any]]:
    """
    Build summary statistics for the observed-only Phase 1 average column using only
    rows with snapshot_confidence greater than the configured threshold.
    """
    filtered_rows = [
        r
        for r in aligned_rows
        if (_safe_numeric(r.get("snapshot_confidence")) is not None)
        and (_safe_numeric(r.get("snapshot_confidence")) > confidence_threshold)
    ]

    percentile_fields = [f"p{p:02d}" for p in percentiles]
    series_specs = [("average_temperature_all_devices", "all_devices_observed")]

    rows: List[Dict[str, Any]] = []
    confidence_label = f"snapshot_confidence_gt_{confidence_threshold:.2f}"

    for column_name, series_name in series_specs:
        values: List[float] = []
        for r in filtered_rows:
            val = _safe_numeric(r.get(column_name))
            if val is not None:
                values.append(val)

        out: Dict[str, Any] = {
            "series_name": series_name,
            "source_column": column_name,
            "confidence_filter": confidence_label,
            "filtered_snapshot_count": len(filtered_rows),
            "non_null_value_count": len(values),
            "mean": "",
            "std_dev": "",
        }
        for f in percentile_fields:
            out[f] = ""

        if values:
            values_sorted = sorted(values)
            out["mean"] = round(statistics.mean(values_sorted), 6)
            out["std_dev"] = (
                round(statistics.stdev(values_sorted), 6)
                if len(values_sorted) > 1
                else 0.0
            )
            for p in percentiles:
                pv = _percentile_linear(values_sorted, p)
                out[f"p{p:02d}"] = "" if pv is None else round(pv, 6)

        rows.append(out)

    return rows











def evaluate_quality_gates(
    qc: Dict[str, Any], gates: Dict[str, Any]
) -> Tuple[bool, List[Dict[str, Any]]]:
    """
    Compare computed QC metrics against gate thresholds from config.
    Returns overall pass/fail and a list of per-gate results.
    """
    checks = [
        {
            "gate": "schema_validity_min_percent",
            "metric": "schema_validity_percent",
            "operator": ">=",
            "actual": float(qc["schema_validity_percent"]),
            "threshold": float(gates["schema_validity_min_percent"]),
        },
        {
            "gate": "duplicate_key_rate_max_percent",
            "metric": "duplicate_key_rate_percent",
            "operator": "<=",
            "actual": float(qc["duplicate_key_rate_percent"]),
            "threshold": float(gates["duplicate_key_rate_max_percent"]),
        },
        {
            "gate": "stale_snapshot_rate_max_percent",
            "metric": "stale_snapshot_rate_percent",
            "operator": "<=",
            "actual": float(qc["stale_snapshot_rate_percent"]),
            "threshold": float(gates["stale_snapshot_rate_max_percent"]),
        },
        {
            "gate": "device_coverage_min_percent",
            "metric": "device_coverage_percent",
            "operator": ">=",
            "actual": float(qc["device_coverage_percent"]),
            "threshold": float(gates["device_coverage_min_percent"]),
        },
    ]

    all_pass = True
    for check in checks:
        if check["operator"] == ">=":
            passed = check["actual"] >= check["threshold"]
        else:
            passed = check["actual"] <= check["threshold"]
        check["passed"] = passed
        all_pass = all_pass and passed

    return all_pass, checks


def append_decisions_log(
    path: Path,
    run_meta: Dict[str, Any],
    gate_checks: List[Dict[str, Any]],
    qc: Dict[str, Any],
) -> None:
    """
    Append a structured markdown section to decisions log.
    This keeps a historical trail of settings and outcomes.
    """
    lines = []
    lines.append("\n## Automated Run")
    lines.append(f"- Date: {run_meta['date_utc']}")
    lines.append(f"- Script: {run_meta['script_name']}")
    lines.append(f"- Config file: {run_meta['config_path']}")
    lines.append(f"- Data directory: {run_meta['data_dir']}")
    lines.append(f"- Files processed: {qc['files_processed']}")
    lines.append(
        "- Excluded non-source CSV filenames: "
        + json.dumps(qc.get("excluded_non_source_csv_filenames", []))
    )
    lines.append("")
    lines.append("### Locked Defaults Used")
    lines.append(f"- bucket_size_seconds: {run_meta['bucket_size_seconds']}")
    lines.append(f"- min_devices_per_snapshot: {run_meta['min_devices_per_snapshot']}")
    lines.append(f"- stale_threshold_seconds: {run_meta['stale_threshold_seconds']}")
    lines.append("")
    lines.append("### Gate Checks")
    for g in gate_checks:
        lines.append(
            f"- {g['gate']}: actual={g['actual']}, threshold {g['operator']} {g['threshold']}, passed={g['passed']}"
        )
    lines.append("")
    lines.append("### QC Snapshot")
    lines.append(f"- schema_validity_percent: {qc['schema_validity_percent']}")
    lines.append(f"- duplicate_key_rate_percent: {qc['duplicate_key_rate_percent']}")
    lines.append(f"- stale_snapshot_rate_percent: {qc['stale_snapshot_rate_percent']}")
    lines.append(f"- device_coverage_percent: {qc['device_coverage_percent']}")

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------






def run_phase1(
    root: Path, *, config_path: Optional[Path] = None,
    data_dir: Optional[Path] = None, limit_files: Optional[int] = None,
    output_dir: Optional[Path] = None,
) -> Dict[str, Any]:
    """Produce the two data files and consolidated producer-owned run record."""
    import os
    import tempfile

    root = root.resolve()
    config_path = (config_path or root / "config" / "transformation_defaults.yaml").resolve()
    data_dir = (data_dir or root / "Data" / "files_csv").resolve()
    cfg = load_yaml_config(config_path)
    keys = ("raw_index", "phase1_aligned")
    production = {key: (root / cfg["artifacts"][key]).resolve() for key in keys}
    if limit_files is not None and (isinstance(limit_files, bool) or not isinstance(limit_files, int) or limit_files <= 0):
        raise ValueError("A smoke limit must be a positive integer.")
    if limit_files is not None and output_dir is None:
        raise ValueError("Limited runs require a separate explicit output directory.")
    if output_dir is None:
        paths = production
        status_path = root / "artifacts" / "phase1_operational" / "phase1_run.json"
    else:
        destination = output_dir.resolve()
        forbidden = [root / name for name in ("scripts", "config", "Data", ".git")]
        forbidden += [root / "artifacts" / name for name in (
            "phase1_outputs", "phase1_operational", "phase1_archive",
            "phase2_outputs", "phase2_operational", "phase2_archive",
            "phase3_outputs", "phase3_operational", "phase3_archive",
        )]
        if (destination == root / "artifacts" or destination == root or root.is_relative_to(destination)
                or any(destination == path.resolve() or destination.is_relative_to(path.resolve()) for path in forbidden)
                or any(path == destination or path.is_relative_to(destination) for path in production.values())):
            raise ValueError("Isolated outputs must not overlap production, source, input or archive paths.")
        paths = {key: destination / production[key].name for key in keys}
        status_path = destination / "phase1_run.json"

    def status(record: Dict[str, Any]) -> None:
        status_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=status_path.parent, delete=False) as stream:
                temporary = Path(stream.name)
                json.dump(record, stream, indent=2, allow_nan=False)
                stream.write("\n")
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temporary, status_path)
        finally:
            if temporary is not None and temporary.exists():
                temporary.unlink()

    status({"schema_version": 1, "status": "in_progress"})
    try:
        files = discover_csv_files(data_dir, limit_files)
        if not files:
            raise ValueError(f"No source CSV files found in {data_dir}.")
        global_cfg = cfg["global"]
        phase1_cfg = cfg["phase1"]
        confidence = cfg["confidence_scoring"]
        index, aligned, qc = build_phase1_temperature(
            files=files, bucket_size_seconds=int(global_cfg["bucket_size_seconds"]),
            stale_threshold_seconds=int(phase1_cfg["stale_threshold_seconds"]),
            min_devices_per_snapshot=int(global_cfg["min_devices_per_snapshot"]),
            drop_snapshot_if_stale_over_threshold=bool(global_cfg.get("drop_snapshot_if_stale_over_threshold", True)),
            completeness_weight=float(confidence["completeness_weight"]),
            freshness_weight=float(confidence["freshness_weight"]),
        )
        qc["excluded_non_source_csv_filenames"] = sorted(
            path.name for path in data_dir.glob("*.csv")
            if path.is_file() and path.name.lower() in NON_SOURCE_CSV_FILENAMES
        )
        statistics_cfg = cfg.get("summary_statistics", {})
        percentiles = sorted({int(p) for p in statistics_cfg.get("percentiles", range(5, 100, 5)) if 0 < int(p) < 100})
        statistics_rows = build_phase1_average_statistics_rows(
            aligned, float(statistics_cfg.get("confidence_threshold", 0.75)), percentiles,
        )
        passed, checks = evaluate_quality_gates(qc, cfg["quality_gates"])
        if not passed:
            raise ValueError(f"Phase 1 quality gates failed: {checks}")
        write_csv_rows(paths["raw_index"], index)
        write_csv_rows(paths["phase1_aligned"], aligned)
        record = {
            "schema_version": 1, "status": "passed", "quality_gates_passed": passed,
            "created_utc": _now_utc_iso(), "qc": qc, "gate_checks": checks,
            "general_statistics": statistics_rows, "settings": {
                "global": global_cfg, "phase1": phase1_cfg,
                "confidence_scoring": confidence, "summary_statistics": statistics_cfg,
                "quality_gates": cfg["quality_gates"],
            },
            "data_dir": str(data_dir), "config_path": str(config_path),
            "artifacts": {key: str(path) for key, path in paths.items()},
            "hashes": {key + "_sha256": _sha256_file(paths[key]) for key in keys},
            "implementation_sha256": _sha256_file(root / "scripts" / "phase1_implement.ipynb"),
            "config_sha256": _sha256_file(config_path),
            "limited_run": limit_files is not None,
        }
        status(record)
        return record
    except Exception as error:
        status({"schema_version": 1, "status": "failed", "error_type": type(error).__name__, "error": str(error)})
        raise


## Optional duplicate investigation definitions

These retain aggregate-inclusive and source-only/reconciliation modes. Defining them does not read raw data or write reports. Expensive investigations are disabled by default.


In [ ]:
#!/usr/bin/env python3
"""Audit raw Phase 1 temperature duplicates without running the transformation."""


import argparse
import csv
import datetime as dt
import json
import math
import re
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path
from typing import Any


def filename_timestamp(name: str) -> dt.datetime | None:
    match = re.search(r"(\d{4}_\d{2}_\d{2}_\d{2}-\d{2}-\d{2})", name)
    if match:
        try:
            return dt.datetime.strptime(
                match.group(1), "%Y_%m_%d_%H-%M-%S"
            ).replace(tzinfo=dt.timezone.utc)
        except ValueError:
            return None
    return None


def discover_audit_csv_files(data_dir: Path) -> list[Path]:
    # Mirror Phase 1 discovery and ordering, including undated filenames first.
    return sorted(
        (path for path in data_dir.glob("*.csv") if path.is_file()),
        key=lambda path: (
            filename_timestamp(path.name) or dt.datetime.min.replace(tzinfo=dt.timezone.utc),
            path.name,
        ),
    )


def percentage(numerator: int, denominator: int) -> float | None:
    return 100.0 * numerator / denominator if denominator else None


def json_number(value: float | None) -> float | str:
    # SQLite stores a parsed NaN as NULL. Never emit non-standard JSON numbers.
    if value is None:
        return "nan"
    return value if math.isfinite(value) else str(value)


def load_observations(
    connection: sqlite3.Connection,
    files: list[Path],
    per_file_counts: list[dict[str, int]] | None = None,
) -> dict[str, int]:
    connection.execute(
        """CREATE TABLE observations (
            time INTEGER NOT NULL, device TEXT NOT NULL, value REAL,
            source INTEGER NOT NULL, row_number INTEGER NOT NULL
        )"""
    )
    counts = dict.fromkeys(
        (
            "rows_total", "rows_schema_valid", "rows_schema_invalid",
            "rows_temperature_raw", "rows_temperature",
            "rows_temperature_invalid", "rows_temperature_nonfinite",
            "files_invalid_schema",
        ),
        0,
    )
    batch: list[tuple[int, str, float, int, int]] = []
    for source, path in enumerate(files):
        previous_counts = counts.copy()
        print(f"Reading {source + 1}/{len(files)}: {path.name}", file=sys.stderr)
        with path.open("r", encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            schema_ok = {"Time", "DeviceId", "Sensor", "Value"}.issubset(
                reader.fieldnames or []
            )
            if not schema_ok:
                counts["files_invalid_schema"] += 1
            for row_number, row in enumerate(reader, start=2):
                counts["rows_total"] += 1
                is_temperature = row.get("Sensor") == "Temperature"
                counts["rows_temperature_raw"] += int(is_temperature)
                if not schema_ok:
                    counts["rows_schema_invalid"] += 1
                    counts["rows_temperature_invalid"] += int(is_temperature)
                    continue
                # Same int(float(Time)), float(Value), and non-empty ID rules
                # as build_phase1_temperature; Sensor is not stripped/case-folded.
                try:
                    timestamp = int(float(row.get("Time", "")))
                    value = float(row.get("Value", ""))
                except (TypeError, ValueError):
                    counts["rows_schema_invalid"] += 1
                    counts["rows_temperature_invalid"] += int(is_temperature)
                    continue
                device = row.get("DeviceId", "")
                if not device or not row.get("Sensor"):
                    counts["rows_schema_invalid"] += 1
                    counts["rows_temperature_invalid"] += int(is_temperature)
                    continue
                counts["rows_schema_valid"] += 1
                if not is_temperature:
                    continue
                counts["rows_temperature"] += 1
                counts["rows_temperature_nonfinite"] += int(not math.isfinite(value))
                batch.append((timestamp, device, value, source, row_number))
                if len(batch) >= 50000:
                    connection.executemany(
                        "INSERT INTO observations VALUES (?, ?, ?, ?, ?)", batch
                    )
                    batch.clear()
        connection.commit()
        if per_file_counts is not None:
            per_file_counts.append(
                {key: counts[key] - previous_counts[key] for key in counts}
            )
    connection.executemany(
        "INSERT INTO observations VALUES (?, ?, ?, ?, ?)", batch
    )
    connection.commit()
    return counts


def analyze(
    connection: sqlite3.Connection,
    files: list[Path],
    counts: dict[str, int],
    max_examples: int,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    print("Grouping temperature keys and source provenance...", file=sys.stderr)
    connection.executescript(
        """
        CREATE INDEX observations_key ON observations(time, device);
        CREATE TABLE duplicates AS
        SELECT time, device, COUNT(*) AS multiplicity,
               COUNT(DISTINCT source) AS file_count,
               COUNT(DISTINCT value) + COUNT(*) - COUNT(value) AS distinct_values,
               COUNT(*) - COUNT(value) AS nan_count,
               MIN(value) AS min_value, MAX(value) AS max_value,
               MAX(value) - MIN(value) AS spread, MIN(rowid) AS first_id
        FROM observations
        GROUP BY time, device HAVING COUNT(*) > 1;
        CREATE UNIQUE INDEX duplicates_key ON duplicates(time, device);
        CREATE TABLE key_sources AS
        SELECT o.time, o.device, o.source, COUNT(*) AS occurrences
        FROM observations AS o JOIN duplicates AS d
        ON o.time = d.time AND o.device = d.device
        GROUP BY o.time, o.device, o.source;
        CREATE UNIQUE INDEX key_sources_key ON key_sources(time, device, source);
        """
    )
    duplicate_count, extra_rows, identical_count, cross_count, mixed_count = (
        connection.execute(
            """SELECT COUNT(*), COALESCE(SUM(multiplicity - 1), 0),
                      COALESCE(SUM(distinct_values = 1 AND nan_count = 0), 0),
                      COALESCE(SUM(file_count > 1), 0),
                      COALESCE(SUM(file_count > 1 AND multiplicity > file_count), 0)
               FROM duplicates"""
        ).fetchone()
    )
    cross_excess, within_excess = connection.execute(
        """SELECT COALESCE(SUM(file_count - 1), 0),
                  COALESCE(SUM(multiplicity - file_count), 0) FROM duplicates"""
    ).fetchone()
    conflict_count = duplicate_count - identical_count
    finite_condition = (
        "distinct_values > 1 AND nan_count = 0 AND spread >= 0 AND spread <= ?"
    )
    finite_count, maximum = connection.execute(
        f"SELECT COUNT(*), MAX(spread) FROM duplicates WHERE {finite_condition}",
        (sys.float_info.max,),
    ).fetchone()
    mean = None
    if finite_count:
        mean = connection.execute(
            f"SELECT SUM(spread / ?) FROM duplicates WHERE {finite_condition}",
            (finite_count, sys.float_info.max),
        ).fetchone()[0]
    near_equal = connection.execute(
        f"SELECT COUNT(*) FROM duplicates WHERE {finite_condition} AND spread <= ?",
        (sys.float_info.max, 1e-9),
    ).fetchone()[0]
    cross_identical = connection.execute(
        """SELECT COUNT(*) FROM duplicates
           WHERE file_count > 1 AND distinct_values = 1 AND nan_count = 0"""
    ).fetchone()[0]
    multiplicities = {
        str(n): count for n, count in connection.execute(
            "SELECT multiplicity, COUNT(*) FROM duplicates GROUP BY multiplicity"
        )
    }
    per_file = [
        {
            "source_file": files[source].name,
            "duplicate_keys": keys,
            "observations_on_duplicate_keys": observations,
            "within_file_excess_rows": within,
        }
        for source, keys, observations, within in connection.execute(
            """SELECT source, COUNT(*), SUM(occurrences), SUM(occurrences - 1)
               FROM key_sources GROUP BY source ORDER BY source"""
        )
    ]
    pairs = [
        {
            "source_files": [files[left].name, files[right].name],
            "shared_duplicate_keys": shared,
        }
        for left, right, shared in connection.execute(
            """SELECT a.source, b.source, COUNT(*) AS shared
               FROM key_sources AS a JOIN key_sources AS b
               ON a.time = b.time AND a.device = b.device AND a.source < b.source
               GROUP BY a.source, b.source
               ORDER BY shared DESC, a.source, b.source LIMIT 20"""
        )
    ]
    examples = []
    for row in connection.execute(
        """SELECT d.*, o.value AS kept_value, o.source AS kept_source,
                  o.row_number AS kept_row
           FROM duplicates AS d JOIN observations AS o ON o.rowid = d.first_id
           WHERE d.distinct_values > 1 OR d.nan_count > 0
           ORDER BY CASE WHEN d.nan_count = 0 AND d.spread <= ?
                         THEN d.spread END DESC, d.time, d.device LIMIT ?""",
        (sys.float_info.max, max_examples),
    ):
        sources = [
            {
                "source_file": files[source].name,
                "occurrences": occurrences,
                "first_row_number": first_row,
                "values": [
                    json_number(value[0])
                    for value in connection.execute(
                        """SELECT value FROM observations
                           WHERE time = ? AND device = ? AND source = ?
                           GROUP BY value ORDER BY MIN(row_number)""",
                        (row["time"], row["device"], source),
                    )
                ],
            }
            for source, occurrences, first_row in connection.execute(
                """SELECT source, COUNT(*), MIN(row_number)
                   FROM observations WHERE time = ? AND device = ?
                   GROUP BY source ORDER BY source""",
                (row["time"], row["device"]),
            )
        ]
        spread = row["spread"]
        examples.append(
            {
                "time_epoch": row["time"], "device_id": row["device"],
                "sensor": "Temperature", "n_occurrences": row["multiplicity"],
                "n_distinct_values": row["distinct_values"],
                "min_value": json_number(row["min_value"]),
                "max_value": json_number(row["max_value"]),
                "abs_diff": (
                    spread if spread is not None and math.isfinite(spread)
                    and not row["nan_count"] else ""
                ),
                "source_files": json.dumps(sources, allow_nan=False),
                "kept_value_phase1": json_number(row["kept_value"]),
                "kept_source_file": files[row["kept_source"]].name,
                "kept_row_number": row["kept_row"],
            }
        )
    summary = {
        **counts,
        "unique_keys": counts["rows_temperature"] - extra_rows,
        "duplicate_rows": extra_rows,
        "duplicate_key_count": duplicate_count,
        "duplicate_key_rate_percent": percentage(extra_rows, counts["rows_temperature"]),
        "duplicate_multiplicity_distribution": multiplicities,
        "identical_value_duplicate_keys": identical_count,
        "identical_value_duplicate_keys_percent": percentage(identical_count, duplicate_count),
        "conflicting_value_duplicate_keys": conflict_count,
        "conflicting_value_duplicate_keys_percent": percentage(conflict_count, duplicate_count),
        "conflicting_value_difference_max": maximum,
        "conflicting_value_difference_mean": mean,
        "conflicting_keys_with_finite_difference": finite_count,
        "conflicting_keys_with_undefined_difference": conflict_count - finite_count,
        "conflicting_keys_with_spread_at_most_1e_9": near_equal,
        "cross_file_duplicate_keys": cross_count,
        "cross_file_duplicate_keys_percent": percentage(cross_count, duplicate_count),
        "cross_file_identical_value_duplicate_keys": cross_identical,
        "within_file_only_duplicate_keys": duplicate_count - cross_count,
        "mixed_cross_and_within_file_duplicate_keys": mixed_count,
        "cross_file_excess_rows": cross_excess,
        "within_file_excess_rows": within_excess,
        "cross_file_excess_rows_percent": percentage(cross_excess, extra_rows),
        "primarily_cross_file_by_key": cross_count > duplicate_count / 2 if duplicate_count else None,
        "primarily_cross_file_by_excess_rows": cross_excess > extra_rows / 2 if extra_rows else None,
        "source_file_duplicate_statistics": per_file,
        "top_20_source_file_pairs": pairs,
        "conflict_examples_written": len(examples),
        "conflict_examples_limit": max_examples,
    }
    return summary, examples


SPREAD_THRESHOLDS = (1e-9, 1e-6, 0.001, 0.01, 0.05, 0.1)


def spread_distribution(connection: sqlite3.Connection) -> dict[str, Any]:
    condition = "distinct_values > 1 AND nan_count = 0 AND spread >= 0 AND spread <= ?"
    exceedances = {}
    bins = []
    lower = 0.0
    for upper in SPREAD_THRESHOLDS:
        count = connection.execute(
            f"SELECT COUNT(*) FROM duplicates WHERE {condition} AND spread > ?",
            (sys.float_info.max, upper),
        ).fetchone()[0]
        exceedances[str(upper)] = count
        bin_count = connection.execute(
            f"""SELECT COUNT(*) FROM duplicates WHERE {condition}
                AND spread > ? AND spread <= ?""",
            (sys.float_info.max, lower, upper),
        ).fetchone()[0]
        bins.append({"lower_exclusive": lower, "upper_inclusive": upper, "keys": bin_count})
        lower = upper
    bins.append(
        {"lower_exclusive": lower, "upper_inclusive": None, "keys": exceedances[str(lower)]}
    )
    return {
        "units": "degrees_Celsius",
        "population": "conflicting duplicate keys with finite max(Value) - min(Value)",
        "strictly_greater_than_counts": exceedances,
        "non_overlapping_bins": bins,
    }


def duplicate_gate(rate: float | None, config_path: Path) -> dict[str, Any]:
    # Only the optional exclusion mode needs the project's existing YAML dependency.
    import yaml

    with config_path.open(encoding="utf-8") as stream:
        config = yaml.safe_load(stream)
    if not isinstance(config, dict) or not isinstance(config.get("quality_gates"), dict):
        raise ValueError(f"Missing quality_gates mapping in {config_path}")
    threshold = config["quality_gates"].get("duplicate_key_rate_max_percent")
    if isinstance(threshold, bool) or not isinstance(threshold, (int, float)):
        raise ValueError(f"Missing or non-numeric duplicate-key rate gate in {config_path}")
    if not math.isfinite(threshold) or threshold < 0:
        raise ValueError(f"Invalid duplicate-key rate gate in {config_path}")
    rounded = round(rate, 6) if rate is not None else None
    return {
        "config_path": str(config_path.resolve()),
        "threshold_percent": threshold,
        "unrounded_rate_percent": rate,
        "strictly_below_threshold": rate < threshold if rate is not None else None,
        "phase1_rounded_rate_percent": rounded,
        "phase1_operator": "<=",
        "phase1_gate_passed": rounded <= threshold if rounded is not None else None,
    }


UNMATCHED_FIELDS = [
    "direction", "time_epoch", "device_id", "sensor", "value",
    "aggregate_count", "timestamped_sources_count", "unmatched_count",
    "aggregate_first_row_number", "timestamped_source_files",
]


def reconcile_records(
    connection: sqlite3.Connection,
    files: list[Path],
    timestamped_ids: list[int],
    aggregate_counts: dict[str, int],
    timestamped_counts: dict[str, int],
    output_path: Path,
) -> dict[str, Any]:
    print("Comparing aggregate and timestamped source record multisets...", file=sys.stderr)
    connection.execute("CREATE TABLE timestamped_sources (source INTEGER PRIMARY KEY)")
    connection.executemany(
        "INSERT INTO timestamped_sources VALUES (?)", ((source,) for source in timestamped_ids)
    )
    connection.executescript(
        """
        CREATE INDEX aggregate_record_key ON aggregate_observations(time, device, value);
        CREATE TABLE record_counts AS
        SELECT time, device, value,
               SUM(is_aggregate) AS aggregate_count,
               COUNT(*) - SUM(is_aggregate) AS source_count
        FROM (
            SELECT time, device, value, 1 AS is_aggregate FROM aggregate_observations
            UNION ALL
            SELECT time, device, value, 0 AS is_aggregate
            FROM observations
            WHERE source IN (SELECT source FROM timestamped_sources)
        )
        GROUP BY time, device, value;
        """
    )
    matched, aggregate_extra, source_extra = connection.execute(
        """SELECT
           COALESCE(SUM(CASE WHEN value IS NULL THEN 0
                            ELSE MIN(aggregate_count, source_count) END), 0),
           COALESCE(SUM(CASE WHEN value IS NULL THEN aggregate_count
                            ELSE MAX(aggregate_count - source_count, 0) END), 0),
           COALESCE(SUM(CASE WHEN value IS NULL THEN source_count
                            ELSE MAX(source_count - aggregate_count, 0) END), 0)
           FROM record_counts"""
    ).fetchone()
    connection.execute(
        """CREATE TABLE unmatched_records AS
           SELECT * FROM record_counts
           WHERE aggregate_count != source_count OR value IS NULL"""
    )
    connection.execute("DROP TABLE record_counts")
    unmatched_groups = {"aggregate_only": 0, "timestamped_sources_only": 0}
    with output_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=UNMATCHED_FIELDS)
        writer.writeheader()
        for row in connection.execute(
            "SELECT * FROM unmatched_records ORDER BY time, device, value"
        ):
            aggregate_count = row["aggregate_count"]
            source_count = row["source_count"]
            aggregate_first_row = connection.execute(
                """SELECT MIN(row_number) FROM aggregate_observations
                   WHERE time = ? AND device = ? AND value IS ?""",
                (row["time"], row["device"], row["value"]),
            ).fetchone()[0]
            sources = [
                {
                    "source_file": files[source].name,
                    "occurrences": occurrences,
                    "first_row_number": first_row,
                }
                for source, occurrences, first_row in connection.execute(
                    """SELECT source, COUNT(*), MIN(row_number) FROM observations
                       WHERE time = ? AND device = ? AND value IS ?
                       AND source IN (SELECT source FROM timestamped_sources)
                       GROUP BY source ORDER BY source""",
                    (row["time"], row["device"], row["value"]),
                )
            ]
            for direction, excess in (
                (
                    "aggregate_only",
                    aggregate_count if row["value"] is None
                    else max(aggregate_count - source_count, 0),
                ),
                (
                    "timestamped_sources_only",
                    source_count if row["value"] is None
                    else max(source_count - aggregate_count, 0),
                ),
            ):
                if not excess:
                    continue
                unmatched_groups[direction] += 1
                writer.writerow(
                    {
                        "direction": direction,
                        "time_epoch": row["time"], "device_id": row["device"],
                        "sensor": "Temperature", "value": json_number(row["value"]),
                        "aggregate_count": aggregate_count,
                        "timestamped_sources_count": source_count,
                        "unmatched_count": excess,
                        "aggregate_first_row_number": aggregate_first_row or "",
                        "timestamped_source_files": json.dumps(sources, allow_nan=False),
                    }
                )
    complete = not any(
        counts["rows_temperature_invalid"] or counts["files_invalid_schema"]
        for counts in (aggregate_counts, timestamped_counts)
    )
    return {
        "aggregate_file": "temperature_raw.csv",
        "aggregate_counts": aggregate_counts,
        "timestamped_source_counts": timestamped_counts,
        "matched_temperature_occurrences": matched,
        "aggregate_unmatched_occurrences": aggregate_extra,
        "timestamped_sources_unmatched_occurrences": source_extra,
        "unmatched_record_groups_by_direction": unmatched_groups,
        "valid_record_multisets_equal": aggregate_extra == 0 and source_extra == 0,
        "verification_complete": complete,
        "all_temperature_records_match": complete and aggregate_extra == 0 and source_extra == 0,
        "definitions": {
            "record": "Phase 1 parsed (Time, DeviceId, Sensor, Value); numeric equality, no tolerance",
            "multiplicity": "counts compared per complete record, without deduplication",
            "nan": "NaN never matches, including another NaN; exported unmatched on each side",
            "infinity": "same-sign infinities match exactly and remain explicitly counted",
            "invalid_rows": "excluded from numeric matching, counted explicitly; prevent complete verification",
            "provenance": "all occurrences per side, not arbitrary attribution of identical excess rows",
        },
    }


EXAMPLE_FIELDS = [
    "time_epoch", "device_id", "sensor", "n_occurrences", "n_distinct_values",
    "min_value", "max_value", "abs_diff", "source_files", "kept_value_phase1",
    "kept_source_file", "kept_row_number",
]


def run_qc(
    data_dir: Path,
    out_dir: Path,
    max_examples: int = 100,
    *,
    exclude_temperature_raw: bool = False,
    config_path: Path | None = None,
) -> dict[str, Any]:
    if max_examples < 0:
        raise ValueError("max_examples must be non-negative")
    if not data_dir.is_dir():
        raise ValueError(f"Data directory does not exist: {data_dir}")
    files = discover_audit_csv_files(data_dir)
    if not files:
        raise ValueError(f"No source CSV files found in {data_dir}")
    aggregate_files = [path for path in files if path.name == "temperature_raw.csv"]
    gate_config = config_path or Path(__file__).resolve().parents[1] / "config" / "transformation_defaults.yaml"
    if exclude_temperature_raw:
        if not aggregate_files:
            raise ValueError(f"Missing aggregate temperature_raw.csv in {data_dir}")
        files = [path for path in files if path.name != "temperature_raw.csv"]
        timestamped_ids = [
            source for source, path in enumerate(files) if filename_timestamp(path.name) is not None
        ]
        if not timestamped_ids:
            raise ValueError(f"No timestamped source CSV files found in {data_dir}")
        duplicate_gate(None, gate_config)
        out_dir.mkdir(parents=True, exist_ok=True)
    # A fresh temporary database bounds memory without retaining raw-data copies.
    with tempfile.TemporaryDirectory(prefix="phase1_duplicate_qc_") as temporary:
        with closing(sqlite3.connect(str(Path(temporary) / "observations.sqlite"))) as connection:
            connection.row_factory = sqlite3.Row
            connection.execute("PRAGMA cache_size = -65536")
            connection.execute("PRAGMA temp_store = FILE")
            if exclude_temperature_raw:
                aggregate_counts = load_observations(connection, aggregate_files)
                connection.execute("ALTER TABLE observations RENAME TO aggregate_observations")
            per_file_counts: list[dict[str, int]] = []
            counts = load_observations(connection, files, per_file_counts)
            summary, examples = analyze(connection, files, counts, max_examples)
            if exclude_temperature_raw:
                summary["conflicting_value_spread_distribution"] = spread_distribution(connection)
                summary["duplicate_row_gate"] = duplicate_gate(
                    summary["duplicate_key_rate_percent"], gate_config
                )
                timestamped_counts = {
                    key: sum(per_file_counts[source][key] for source in timestamped_ids)
                    for key in counts
                }
                summary["aggregate_reconciliation"] = reconcile_records(
                    connection, files, timestamped_ids, aggregate_counts,
                    timestamped_counts, out_dir / "qc_phase1_unmatched_records.csv",
                )
                summary["file_inventory"] = {
                    "excluded_from_duplicate_statistics": ["temperature_raw.csv"],
                    "timestamped_sources": [files[source].name for source in timestamped_ids],
                    "other_files_in_duplicate_statistics_only": [
                        path.name for path in files if filename_timestamp(path.name) is None
                    ],
                }
    summary.update(
        {
            "generated_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "data_dir": str(data_dir.resolve()),
            "files_processed": len(files),
            "definitions": {
                "key": ["int(float(Time))", "DeviceId", "Sensor == Temperature"],
                "temperature_rows": "Phase 1 schema-valid temperature rows before deduplication",
                "duplicate_rows": "sum(multiplicity - 1); first occurrence is retained by Phase 1",
                "identical_values": "exact parsed-float equality, no tolerance; NaN is never equal",
                "value_difference": "max(Value) - min(Value) per conflicting key; mean is unweighted",
                "nonfinite_values": "accepted as in Phase 1; undefined spreads excluded and counted",
                "zero_denominator": "percentages and differences with no applicable keys are null",
                "cross_file_excess_rows": "sum(distinct source files - 1) per duplicated key",
                "within_file_excess_rows": "sum(multiplicity - distinct source files) per duplicated key",
                "primarily": "strictly more than 50 percent, assessed by keys and excess rows",
                "example_order": "descending spread, then Time and DeviceId; undefined spreads last",
                "row_number": "CSV record number including header as record 1",
            },
        }
    )
    out_dir.mkdir(parents=True, exist_ok=True)
    with (out_dir / "qc_phase1_duplicate_conflicts.csv").open(
        "w", encoding="utf-8", newline=""
    ) as stream:
        writer = csv.DictWriter(stream, fieldnames=EXAMPLE_FIELDS)
        writer.writeheader()
        writer.writerows(examples)
    with (out_dir / "qc_phase1_duplicate_summary.json").open("w", encoding="utf-8") as stream:
        json.dump(summary, stream, indent=2, allow_nan=False)
        stream.write("\n")
    return summary




def main() -> None:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=root / "Data" / "files_csv")
    parser.add_argument(
        "--out-dir", type=Path, required=True,
        help="Output directory (defaults to a mode-specific artifacts subfolder).",
    )
    parser.add_argument(
        "--exclude-temperature-raw", action="store_true",
        help="Exclude temperature_raw.csv from duplicates and verify it against timestamped sources.",
    )
    parser.add_argument(
        "--config", type=Path, default=root / "config" / "transformation_defaults.yaml",
        help="Read-only Phase 1 QC gate configuration for exclusion mode.",
    )
    parser.add_argument("--max-examples", type=int, default=100)
    args = parser.parse_args()
    if args.max_examples < 0:
        parser.error("--max-examples must be non-negative")
    if not args.data_dir.is_dir():
        parser.error(f"Data directory does not exist: {args.data_dir}")
    out_dir = args.out_dir or root / "artifacts" / "phase1_duplicate_qc"
    if args.exclude_temperature_raw and args.out_dir is None:
        out_dir = out_dir / "excluding_temperature_raw"
    summary = run_qc(
        args.data_dir, out_dir, args.max_examples,
        exclude_temperature_raw=args.exclude_temperature_raw, config_path=args.config,
    )
    print(json.dumps(
        {key: value for key, value in summary.items()
         if key not in {
             "source_file_duplicate_statistics", "top_20_source_file_pairs",
             "definitions", "file_inventory",
         }},
        indent=2, allow_nan=False,
    ))
    print(f"Reports written to {out_dir.resolve()}")




## Configuration and run mode

`LIMIT_FILES = None` runs production in `artifacts/phase1_operational/`. A limited smoke run requires a separate explicit output directory and cannot overwrite production. The run JSON contains settings, general statistics, QC and producer status.


In [ ]:
ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "transformation_defaults.yaml").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run from the sensor_project repository or scripts directory.")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.shared_monitoring_engine import load_phase1_definitions

LIMIT_FILES = None
OUTPUT_DIR = None
RUN_DUPLICATE_AUDIT = False
EXCLUDE_AGGREGATE_FROM_AUDIT = False
AUDIT_OUTPUT_DIR = None


## Transformation, publication, and quality gates

Failures raise and leave explicit failed status. Downstream workflows reject an incomplete or failed production publication. Run the independent tests after implementation.


In [ ]:
phase1_result = run_phase1(ROOT, limit_files=LIMIT_FILES, output_dir=OUTPUT_DIR)
print(json.dumps(phase1_result, indent=2, allow_nan=False))


In [ ]:
if RUN_DUPLICATE_AUDIT:
    if AUDIT_OUTPUT_DIR is None:
        raise ValueError("Choose an explicit audit output directory; historical reports must not be overwritten.")
    audit = load_phase1_definitions(ROOT, audit=True)
    audit_result = audit.run_qc(
        ROOT / "Data" / "files_csv", Path(AUDIT_OUTPUT_DIR),
        exclude_temperature_raw=EXCLUDE_AGGREGATE_FROM_AUDIT,
        config_path=ROOT / "config" / "transformation_defaults.yaml",
    )
    print(json.dumps({key: value for key, value in audit_result.items() if key != "source_file_duplicate_statistics"}, indent=2, allow_nan=False))
